In [1]:
import os
import pandas as pd

In [2]:
os.chdir('S:/LLC_0028/data/')

In [3]:
dta = pd.read_csv(f'./harmonised_all/llc_0028_full_harmonised_data_all_v2.csv', index_col=0)

In [4]:
core_symp = ['fever','cough','throat',
            'chest_tight','breath','nose','aches',
             'fatigue', 'diarrhoea','smell_taste',
            'nausea_vomit', 'sneezing',
             'headache','concentrating','memory']

#remove studies that didnt ask about all core_symp
dta = dta.dropna(how='any',subset=core_symp)

dta_symp = dta.loc[~(dta[core_symp]==0).all(axis=1)]
dta_asymp = dta.loc[(dta[core_symp]==0).all(axis=1)]

In [5]:
sociodemo = ['llc_age','llc_ethnic3','llc_sex','functional_limitation_cat']
columns = ['LLC_0028_stud_id','study','covid_status'] + sociodemo

In [6]:
dta = dta[columns]

In [8]:
dta.shape

(26144, 7)

In [9]:
ethnicity_encoded = pd.get_dummies(dta.llc_ethnic3, 
                                   dtype=float, prefix='ethnicity',
                                  dummy_na=True)

In [10]:
dta = dta.merge(ethnicity_encoded, left_index=True, right_index=True, how='left')

In [11]:
dta['ethnicity_missing'] = list(dta['ethnicity_99.0'] + dta['ethnicity_nan'])

In [12]:
fl_encoded = pd.get_dummies(dta.functional_limitation_cat, 
                                   dtype=float, prefix='fl',
                                  dummy_na=False)

In [13]:
dta = dta.merge(fl_encoded, left_index=True, right_index=True, how='left')

In [14]:
age_bins = pd.cut(dta.llc_age, bins = [18,45,55,65,100], labels = ['18-45','45-55','55-65','65+'])

In [15]:
dta['age_cat'] = age_bins

In [16]:
age_encoded = pd.get_dummies(dta.age_cat,dtype=int,dummy_na=True,prefix='age')

dta = dta.merge(age_encoded, left_index=True, right_index=True,how='left')

In [17]:
study_encoded = pd.get_dummies(dta.study,dtype=int,prefix='study')
dta = dta.merge(study_encoded, left_index=True, right_index=True, how='left')

In [18]:
dta = dta.drop(['ethnicity_99.0','ethnicity_nan',
                'llc_ethnic3','functional_limitation_cat',
                'llc_age','age_cat', 'study'], axis=1)

In [37]:
def create_table(dta, dta_asymp):
    
    pop_stats = pd.DataFrame(index = list(dta.columns[1:]) + ['asymptomatic', 'sample size'])

    for status, group in dta.groupby('covid_status'):
        
        values = []
        
        for column in group:
            
            if column in ['LLC_0028_stud_id']:
                continue
            if column =='covid_status':
                values.append(status)
            else:
                count = round(group[column].sum()/5,0)*5
                perc = round(count*100/group.shape[0],0)
                
                if count>=10:
                    values.append(f'{int(count)} ({perc})')
                else:
                    values.append('<10 (-)')
        #add asymptomatic
        count = round(dta_asymp.loc[dta_asymp.covid_status==status].shape[0]/5,0)*5
        perc = round(count*100/dta.loc[dta.covid_status==status].shape[0],0)
        values.append(f'{int(count)} ({perc})')
        
        #add sample size
        count = round(group.shape[0],0)
        perc = round(count*100/dta.shape[0],0)
        values.append(f'{int(count)} ({perc})')
        # add to dataframe
        pop_stats[str(status)] = values
            
            
    #repeat for full population
    values = []
    for column in dta:
            
        if column in ['LLC_0028_stud_id']:
            continue
        if column =='covid_status':
            values.append('all')
        else:
            try:
                v1 = int(pop_stats.loc[column,'1.0'].split(' ')[0])
            except ValueError:
                v1=0
                
            v2 = int(pop_stats.loc[column,'2.0'].split(' ')[0])
            
            try:
                v3 = int(pop_stats.loc[column,'0.0'].split(' ')[0])
            except ValueError:
                v3=0
                
            count = v1+v2+v3
            perc = round(count*100/dta.shape[0],0)
                
            if count>=10:
                values.append(f'{int(count)} ({perc})')
            else:
                values.append('<10 (-)')
    #add asymptomatic
    column = 'asymptomatic'
    try:
        v1 = int(pop_stats.loc[column,'1.0'].split(' ')[0])
    except ValueError:
        v1=0
                
    v2 = int(pop_stats.loc[column,'2.0'].split(' ')[0])
            
    try:
        v3 = int(pop_stats.loc[column,'0.0'].split(' ')[0])
    except ValueError:
        v3=0
                
    count = v1+v2+v3
    perc = round(count*100/dta.shape[0],0)

    values.append(f'{int(count)} ({perc})')
    
    #add sample size
    count = dta.shape[0]
    perc = round(count*100/dta.shape[0],0)
    values.append(f'{int(count)} ({perc})')
    
    pop_stats['all'] = values
    
    #disclosure control on FL values
    for idx in ['fl_2','fl_3','fl_4']:
    
        v1 = int(pop_stats.loc[idx,'1.0'].split(' ')[0])
        v2 = int(pop_stats.loc[idx,'2.0'].split(' ')[0])
        vall = v1+v2
        pop_stats.loc[idx,'all'] = f'{vall} '+pop_stats.loc[idx,'all'].split(' ')[1]
    
    return pop_stats

In [38]:
pop_stats = create_table(dta, dta_asymp)

In [40]:
pop_stats = pop_stats.rename(index = {'age_18-45':'Age: 18-45, N (%)',
                                      'age_45-55':'Age: 45-55, N (%)',
                                      'age_55-65':'Age: 55-65, N (%)',
                                      'age_65+':'Age: 65+, N (%)',
                                      'age_nan':'Age: unknown, N (%)',
                                      'llc_sex':'Female, N (%)',
                                      'ethnicity_0.0': 'Ethnicity: white, N (%)',
                                      'ethnicity_7.0': 'Ethnicity: other, N (%)',
                                      'ethnicity_missing': 'Ethnicity: unknown, N (%)',
                                      'fl_-1':'Functional limitation: unknown, N (%)',
                                      'fl_0':'Functional limitation: none, N (%)',
                                     'fl_1':'Functional limitation: < 2 weeks, N (%)',
                                     'fl_2':'Functional limitation: 2-4 weeks, N (%)',
                                     'fl_3':'Functional limitation: 4-12 weeks, N (%)',
                                     'fl_4':'Functional limitation: 12+ weeks, N (%)',
                                     'sample size': 'Sample size, N (%)',
                                     'asymptomatic': 'Asymptomatic, N(%)'},
                            columns = {'0.0':'No covid',
                                       '1.0': 'Covid < 12 weeks ago',
                                       '2.0': 'Covid > 12 weeks ago',
                                       'all': 'All participants'})

study_dict = dict(zip([c for c in dta.columns if 'study' in c], 
                      [c.split('_')[1] for c in dta.columns if 'study' in c]))

pop_stats = pop_stats.rename(index = study_dict)

In [42]:
pop_stats.to_csv(f'./harmonised_all/llc_0028_population_demog_data_v2.csv')